# 🎵 VibeFinder — Spotify Mood & Energy Analyzer

Pulls your Spotify listening history, fetches audio features per track (energy, valence, danceability), classifies each song into a mood, and tells you what your music says about your headspace.

**Stack:** Python · Spotify Web API · Pandas · AWS S3  
**Auth:** Free Spotify Developer account (developer.spotify.com)

## 1. Imports & Config

In [ ]:
import requests
import base64
import pandas as pd
from datetime import datetime

# Add your Spotify app credentials here
# Get them free at: developer.spotify.com → Create App
SPOTIFY_CLIENT_ID     = "your_client_id_here"
SPOTIFY_CLIENT_SECRET = "your_client_secret_here"

print("Config ready ✓")

## 2. Sample Data

Realistic Spotify audio features for 15 real tracks. Once you add your credentials above, replace this with a live API call using `SpotifyClient` further below.

In [ ]:
SAMPLE_TRACKS = [
    {"id": "t01", "name": "Blinding Lights",      "artist": "The Weeknd",        "energy": 0.73, "valence": 0.33, "danceability": 0.51, "tempo": 171.0, "played_at": "2024-01-20T08:15:00"},
    {"id": "t02", "name": "As It Was",             "artist": "Harry Styles",      "energy": 0.73, "valence": 0.66, "danceability": 0.52, "tempo": 124.0, "played_at": "2024-01-20T08:20:00"},
    {"id": "t03", "name": "Flowers",               "artist": "Miley Cyrus",       "energy": 0.66, "valence": 0.91, "danceability": 0.76, "tempo": 118.0, "played_at": "2024-01-20T08:25:00"},
    {"id": "t04", "name": "Anti-Hero",             "artist": "Taylor Swift",      "energy": 0.64, "valence": 0.53, "danceability": 0.64, "tempo": 97.0,  "played_at": "2024-01-20T12:00:00"},
    {"id": "t05", "name": "Cruel Summer",          "artist": "Taylor Swift",      "energy": 0.70, "valence": 0.56, "danceability": 0.55, "tempo": 170.0, "played_at": "2024-01-20T12:05:00"},
    {"id": "t06", "name": "ghost in the machine",  "artist": "SZA",               "energy": 0.39, "valence": 0.25, "danceability": 0.63, "tempo": 78.0,  "played_at": "2024-01-20T22:00:00"},
    {"id": "t07", "name": "Kill Bill",             "artist": "SZA",               "energy": 0.40, "valence": 0.27, "danceability": 0.73, "tempo": 89.0,  "played_at": "2024-01-20T22:05:00"},
    {"id": "t08", "name": "Calm Down",             "artist": "Rema & Selena",     "energy": 0.80, "valence": 0.72, "danceability": 0.85, "tempo": 107.0, "played_at": "2024-01-21T09:00:00"},
    {"id": "t09", "name": "Escapism",              "artist": "RAYE",              "energy": 0.59, "valence": 0.34, "danceability": 0.74, "tempo": 105.0, "played_at": "2024-01-21T09:10:00"},
    {"id": "t10", "name": "Lift Me Up",            "artist": "Rihanna",           "energy": 0.23, "valence": 0.11, "danceability": 0.31, "tempo": 67.0,  "played_at": "2024-01-21T23:00:00"},
    {"id": "t11", "name": "Made You Look",         "artist": "Meghan Trainor",    "energy": 0.87, "valence": 0.96, "danceability": 0.80, "tempo": 116.0, "played_at": "2024-01-22T07:30:00"},
    {"id": "t12", "name": "unholy",                "artist": "Sam Smith",         "energy": 0.55, "valence": 0.37, "danceability": 0.68, "tempo": 131.0, "played_at": "2024-01-22T18:00:00"},
    {"id": "t13", "name": "Rich Flex",             "artist": "Drake & 21 Savage", "energy": 0.60, "valence": 0.43, "danceability": 0.84, "tempo": 93.0,  "played_at": "2024-01-23T11:00:00"},
    {"id": "t14", "name": "Sunroof",               "artist": "Nicky Youre",       "energy": 0.65, "valence": 0.95, "danceability": 0.72, "tempo": 120.0, "played_at": "2024-01-23T14:00:00"},
    {"id": "t15", "name": "Running Up That Hill",  "artist": "Kate Bush",         "energy": 0.62, "valence": 0.25, "danceability": 0.42, "tempo": 103.0, "played_at": "2024-01-23T21:00:00"},
]

## 3. Spotify API Client

Handles auth and fetching audio features. Skip to Section 4 if using sample data.

In [ ]:
class SpotifyClient:
    BASE_URL = "https://api.spotify.com/v1"

    def __init__(self, client_id, client_secret):
        self.client_id = client_id
        self.client_secret = client_secret
        self.token = None

    def authenticate(self):
        credentials = base64.b64encode(
            f"{self.client_id}:{self.client_secret}".encode()
        ).decode()
        resp = requests.post(
            "https://accounts.spotify.com/api/token",
            headers={"Authorization": f"Basic {credentials}"},
            data={"grant_type": "client_credentials"},
        )
        self.token = resp.json().get("access_token")
        print("Authenticated with Spotify ✓")

    def get_audio_features(self, track_ids: list) -> list:
        ids_str = ",".join(track_ids[:100])
        resp = requests.get(
            f"{self.BASE_URL}/audio-features",
            headers={"Authorization": f"Bearer {self.token}"},
            params={"ids": ids_str},
        )
        return resp.json().get("audio_features", [])

# Uncomment to use live data:
# client = SpotifyClient(SPOTIFY_CLIENT_ID, SPOTIFY_CLIENT_SECRET)
# client.authenticate()
# features = client.get_audio_features([t["id"] for t in SAMPLE_TRACKS])

print("SpotifyClient defined ✓")

## 4. Mood Classification

Two axes define mood — same quadrant model Spotify uses internally:

```
              HIGH ENERGY
                  │
    😤 Intense    │    🔥 Pump Up
────────────────────────────────── VALENCE (happiness →)
    🌧️ Melancholy │   😌 Chill & Happy
                  │
              LOW ENERGY
```

In [ ]:
def classify_mood(energy: float, valence: float) -> str:
    if energy >= 0.6 and valence >= 0.6:   return "🔥 Pump Up"
    elif energy >= 0.6 and valence < 0.6:  return "😤 Intense"
    elif energy < 0.6 and valence >= 0.6:  return "😌 Chill & Happy"
    else:                                   return "🌧️ Melancholy"

def classify_time_of_day(played_at: str) -> str:
    hour = datetime.fromisoformat(played_at).hour
    if 5 <= hour < 12:   return "Morning"
    if 12 <= hour < 17:  return "Afternoon"
    if 17 <= hour < 21:  return "Evening"
    return "Night"

print("Mood classifiers ready ✓")

## 5. Build the DataFrame

In [ ]:
df = pd.DataFrame(SAMPLE_TRACKS)
df["mood"]         = df.apply(lambda r: classify_mood(r["energy"], r["valence"]), axis=1)
df["time_of_day"]  = df["played_at"].apply(classify_time_of_day)

print(f"{len(df)} tracks loaded")
df[["name", "artist", "energy", "valence", "mood", "time_of_day"]]

## 6. Vibe Score

In [ ]:
score = round((df["energy"].mean() * 0.5 + df["valence"].mean() * 0.5) * 100)

if score >= 65:    label = "You're in a great headspace 🙌"
elif score >= 45:  label = "Balanced — a little of everything 🎵"
else:              label = "Introspective mode 🌙"

print(f"✨ Your Vibe Score: {score}/100")
print(f"   {label}")

## 7. Mood Distribution

In [ ]:
mood_dist = (
    df.groupby("mood")
    .agg(tracks=("name", "count"),
         avg_energy=("energy", "mean"),
         avg_valence=("valence", "mean"))
    .reset_index()
    .sort_values("tracks", ascending=False)
    .round(2)
)
mood_dist

## 8. Listening by Time of Day

In [ ]:
time_analysis = (
    df.groupby("time_of_day")
    .agg(tracks=("name", "count"),
         dominant_mood=("mood", lambda x: x.value_counts().index[0]),
         avg_energy=("energy", "mean"))
    .reindex(["Morning", "Afternoon", "Evening", "Night"])
    .dropna()
    .reset_index()
    .round(2)
)
time_analysis

## 9. Top Artists

In [ ]:
top_artists = (
    df.groupby("artist")
    .agg(plays=("name", "count"),
         avg_energy=("energy", "mean"),
         avg_valence=("valence", "mean"))
    .reset_index()
    .sort_values("plays", ascending=False)
    .head(5)
    .round(2)
)
top_artists

## 10. (Optional) Save to AWS S3

In [ ]:
# Uncomment and configure to save your vibe history to S3

# import boto3
# from io import StringIO
#
# S3_BUCKET = "your-vibefinder-bucket"
# date_str = datetime.utcnow().strftime("%Y/%m/%d")
# key = f"listening_history/{date_str}/vibe_snapshot.csv"
#
# s3 = boto3.client("s3")
# buf = StringIO()
# df.to_csv(buf, index=False)
# s3.put_object(Bucket=S3_BUCKET, Key=key, Body=buf.getvalue())
# print(f"Saved to s3://{S3_BUCKET}/{key} ✓")

print("S3 save block ready — uncomment to activate")